# Lesson 1.1 — Robot state and observation

The question this notebook answers:

> when the environment returns `obs`, what exactly is in it, and which part of it
> could a real robot actually measure?

Two words that are often used interchangeably and must not be:

- **state** — everything the simulator knows about the world. May include
  quantities no real sensor provides.
- **observation** — what the policy is handed. It is a *choice*, made by the
  `obs_mode` argument, about which part of the state is exposed and in what shape.

Run the cells in order. The interesting step is section 1.1.4, where a naive guess
about the layout turns out to be wrong.

## 1.1.1 — Create the environment

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

env = make_env(obs_mode="state")
print(env)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)

2026-09-22 11:36:38,088 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>
observation space: Box(-inf, inf, (1, 42), float32)
action space     : Box(-1.0, 1.0, (8,), float32)


## 1.1.2 — The structured observation: `state_dict`

`obs_mode="state"` returns a flat tensor, which hides its structure. The same
information is available unflattened through `obs_mode="state_dict"`, so switch
modes and look at the tree before trusting any index.

In [2]:
env_sd = make_env(obs_mode="state_dict")
obs_sd, info = env_sd.reset(seed=0)


def print_tree(data, prefix=""):
    """Print the nested observation structure with tensor shapes."""
    if isinstance(data, dict):
        for key, value in data.items():
            name = f"{prefix}.{key}" if prefix else key
            print_tree(value, name)
    else:
        print(f"  {prefix:<28} shape={getattr(data, 'shape', None)} dtype={getattr(data, 'dtype', None)}")


print("observation tree:")
print_tree(obs_sd)

2026-09-22 11:36:38,466 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


observation tree:
  agent.qpos                   shape=torch.Size([1, 9]) dtype=torch.float32
  agent.qvel                   shape=torch.Size([1, 9]) dtype=torch.float32
  extra.is_grasped             shape=torch.Size([1]) dtype=torch.bool
  extra.tcp_pose               shape=torch.Size([1, 7]) dtype=torch.float32
  extra.goal_pos               shape=torch.Size([1, 3]) dtype=torch.float32
  extra.obj_pose               shape=torch.Size([1, 7]) dtype=torch.float32
  extra.tcp_to_obj_pos         shape=torch.Size([1, 3]) dtype=torch.float32
  extra.obj_to_goal_pos        shape=torch.Size([1, 3]) dtype=torch.float32


## 1.1.3 — The flattened observation, and `proprioception`

Policy inputs are usually flat vectors, so the structure above has to be
concatenated into one tensor. ManiSkill also exposes robot-only quantities
through `get_proprioception()` and `get_state()`, which is how you separate
"what the robot knows about itself" from "what the task adds".

In [3]:
obs, info = env.reset(seed=0)
flat = obs[0]
print("flat observation shape:", flat.shape, "dtype:", flat.dtype)
print("first 9 values (joint positions?):", flat[:9])

agent = env.unwrapped.agent


def describe(value, prefix="", depth=0):
    """Describe a nested dict of tensors, since these helpers return nested dicts."""
    if isinstance(value, dict):
        for key, inner in value.items():
            name = f"{prefix}.{key}" if prefix else key
            describe(inner, name, depth + 1)
    else:
        shape = getattr(value, "shape", None)
        print(f"  {prefix:<24} type={type(value).__name__:<10} shape={shape}")


proprio = agent.get_proprioception()
print("\n-- get_proprioception() --")
print("top-level keys:", list(proprio.keys()))
describe(proprio)

state = agent.get_state()
print("\n-- agent.get_state() --")
print("top-level keys:", list(state.keys()))
describe(state)

print("\n-- end-effector --")
print("TCP position:", agent.tcp_pos)
print("TCP pose (7 values, position + quaternion):", agent.tcp_pose.raw_pose[0])

print("\nThese helpers return nested dicts, not flat tensors: there is no single .shape.")

flat observation shape: torch.Size([42]) dtype: torch.float32
first 9 values (joint positions?): tensor([ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
         0.0400])

-- get_proprioception() --
top-level keys: ['qpos', 'qvel']
  qpos                     type=Tensor     shape=torch.Size([1, 9])
  qvel                     type=Tensor     shape=torch.Size([1, 9])

-- agent.get_state() --
top-level keys: ['robot_root_pose', 'robot_root_vel', 'robot_root_qvel', 'robot_qpos', 'robot_qvel', 'controller']
  robot_root_pose          type=Pose       shape=torch.Size([1, 7])
  robot_root_vel           type=Tensor     shape=torch.Size([1, 3])
  robot_root_qvel          type=Tensor     shape=torch.Size([1, 3])
  robot_qpos               type=Tensor     shape=torch.Size([1, 9])
  robot_qvel               type=Tensor     shape=torch.Size([1, 9])

-- end-effector --
TCP position: tensor([[0.0123, 0.0380, 0.1822]])
TCP pose (7 values, position + quaternion): tensor([ 0.0123

## 1.1.4 — Predict the 42-d layout, then verify it

**Before running the next cell**, write down your prediction. The obvious guess is
to concatenate the `state_dict` components in the order the tree printed them:

```text
qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) + obj_pose(7)
        + tcp_to_obj_pos(3) + obj_to_goal_pos(3) + is_grasped(1)
```

That guess is **wrong**. The flattened vector uses a different member order than
`state_dict` insertion order. Comparing the two makes the trap concrete: if you
index by assumption rather than by verification, every downstream feature is
silently mislabeled.

> In a live session this specific mismatch was observed directly: concatenating
> in `state_dict` order and comparing with `obs_mode="state"` returned
> `False` from `np.allclose`.

In [4]:
from mani_skill.utils import common


def as_flat(tensor):
    return tensor.reshape(-1).cpu().numpy()


# Candidate A: concatenate in state_dict insertion order
guess_a = np.concatenate([
    as_flat(obs_sd["agent"]["qpos"]),
    as_flat(obs_sd["agent"]["qvel"]),
    as_flat(obs_sd["extra"]["tcp_pose"]),
    as_flat(obs_sd["extra"]["goal_pos"]),
    as_flat(obs_sd["extra"]["obj_pose"]),
    as_flat(obs_sd["extra"]["tcp_to_obj_pos"]),
    as_flat(obs_sd["extra"]["obj_to_goal_pos"]),
    as_flat(obs_sd["extra"]["is_grasped"]),
])

# Candidate B: the layout actually used by obs_mode="state"
guess_b = np.concatenate([
    as_flat(obs_sd["agent"]["qpos"]),
    as_flat(obs_sd["agent"]["qvel"]),
    as_flat(obs_sd["extra"]["is_grasped"]),
    as_flat(obs_sd["extra"]["tcp_pose"]),
    as_flat(obs_sd["extra"]["goal_pos"]),
    as_flat(obs_sd["extra"]["obj_pose"]),
    as_flat(obs_sd["extra"]["tcp_to_obj_pos"]),
    as_flat(obs_sd["extra"]["obj_to_goal_pos"]),
])

flat_state = as_flat(obs[0])
print("state_dict-order matches state:", np.allclose(guess_a, flat_state, atol=1e-6))
print("verified-order  matches state:", np.allclose(guess_b, flat_state, atol=1e-6))

state_dict-order matches state: False
verified-order  matches state: True


### The verified 42-d layout

| Slice | Field | Dim |
|---|---|---:|
| `[0:9]` | `agent.qpos` | 9 |
| `[9:18]` | `agent.qvel` | 9 |
| `[18:19]` | `extra.is_grasped` | 1 |
| `[19:26]` | `extra.tcp_pose` | 7 |
| `[26:29]` | `extra.goal_pos` | 3 |
| `[29:36]` | `extra.obj_pose` | 7 |
| `[36:39]` | `extra.tcp_to_obj_pos` | 3 |
| `[39:42]` | `extra.obj_to_goal_pos` | 3 |

Note where `is_grasped` sits: at index 18, wedged between `qvel` and `tcp_pose`,
not at the end. This is the same layout documented in
`archive/lesson_0_1/verify_state_flattening.py` and `build_deployment_safe_observation.py`.

`tcp_pose` and `obj_pose` are 7-dimensional: 3 position values followed by a
4-value quaternion (see `1.3_coordinate_frames.ipynb` for the component order).

## 1.1.5 — Deployable versus privileged state

The 42-d vector mixes two very different things:

- **deployable**: joint positions, joint velocities, TCP pose. A real arm with
  encoders and forward kinematics can produce these.
- **privileged**: object pose, grasp state, and the relative vectors — a
  simulator can read them exactly, a real setup usually cannot without perception.

`scripts/pipeline/observation_adapter.py` encodes this split. Reproducing it here
shows why a policy trained on all 42 dimensions cannot be deployed as-is.

In [5]:
import sys
from pathlib import Path

# The observation adapter is a shared module, not part of the conversion pipeline:
# identifying which state fields are deployable is a modelling decision, not a
# conversion step.
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from observation_adapter import (
    build_deployment_safe_observation,
    build_privileged_observation,
)

deployable = build_deployment_safe_observation(obs_sd)
privileged = build_privileged_observation(obs_sd)

print("deployable dim:", deployable.shape, " = qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) = 28")
print("privileged dim:", privileged.shape, " = is_grasped(1) + obj_pose(7) + tcp_to_obj(3) + obj_to_goal(3) = 14")
print("28 + 14 =", deployable.shape[-1] + privileged.shape[-1])
print("\ndeployable excludes:", ["is_grasped", "obj_pose", "tcp_to_obj_pos", "obj_to_goal_pos"])

deployable dim: torch.Size([1, 28])  = qpos(9) + qvel(9) + tcp_pose(7) + goal_pos(3) = 28
privileged dim: torch.Size([1, 14])  = is_grasped(1) + obj_pose(7) + tcp_to_obj(3) + obj_to_goal(3) = 14
28 + 14 = 42

deployable excludes: ['is_grasped', 'obj_pose', 'tcp_to_obj_pos', 'obj_to_goal_pos']


## Takeaways

1. `obs_mode` decides what the policy sees; the underlying state is larger.
2. The flattened layout is **not** the `state_dict` insertion order. Verify
   offsets empirically; never infer them from a printed tree.
3. `proprioception` / `get_state()` separate robot-side quantities from
   task-side ones.
4. 42 dimensions contain 14 dimensions of simulator-privileged information. Any
   claim that a state-based policy is deployable has to answer what replaces
   those 14 values on real hardware.

Next: `1.2_action_space_and_control_modes.ipynb`.